In [41]:
import numpy as np
import sympy as sy
from scipy import signal
from scipy import datasets
from matplotlib import pyplot as plt

In [42]:
from naive_convolve import naive_convolve

In [43]:
import convolution_matrices
from utils import plot_pdf, symmetrical_cyclic_convolution

In [44]:
data = np.arange(5*5).reshape(5, 5)
data

array([[ 0,  1,  2,  3,  4],
       [ 5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14],
       [15, 16, 17, 18, 19],
       [20, 21, 22, 23, 24]])

In [45]:
kernel = np.array([
    [ 1, 2, 1],
    [ 2, 1, 2],
    [ 1, 2, 1],
])

In [46]:
output = signal.convolve2d(data, kernel, mode='valid')

In [47]:
output

array([[ 78,  91, 104],
       [143, 156, 169],
       [208, 221, 234]])

In [48]:
output_naive = naive_convolve(data, kernel)

In [49]:
output_naive

array([[ 78,  91, 104],
       [143, 156, 169],
       [208, 221, 234]])

In [50]:
d_num = 3
g_num = 3

In [51]:
di = sy.Matrix(sy.symbols(" ".join(f"d_{i}"for i in range(d_num))))
di

Matrix([
[d_0],
[d_1],
[d_2]])

In [52]:
gi = sy.Matrix(sy.symbols(" ".join(f"g_{i}"for i in range(g_num))))
gi

Matrix([
[g_0],
[g_1],
[g_2]])

In [53]:
params = convolution_matrices.c3x3m4a11e2()

In [54]:
a_mtx = sy.Matrix(params["a"])
a_mtx

Matrix([
[1, 1,  1],
[1, 0, -1],
[0, 1, -1],
[1, 1, -2]])

In [55]:
b_mtx = sy.Matrix(params["b"])
b_mtx

Matrix([
[1, 1,  1],
[3, 0, -3],
[0, 3, -3],
[1, 1, -2]])

In [56]:
g_mtx = sy.diag(*([sy.Rational(*params["g"])]*(max(b_mtx.shape))))
g_mtx

Matrix([
[1/3,   0,   0,   0],
[  0, 1/3,   0,   0],
[  0,   0, 1/3,   0],
[  0,   0,   0, 1/3]])

In [57]:
bg_mtx = sy.diag(*(g_mtx*b_mtx*gi).tolist())
bg_mtx

Matrix([
[g_0/3 + g_1/3 + g_2/3,         0,         0,                       0],
[                    0, g_0 - g_2,         0,                       0],
[                    0,         0, g_1 - g_2,                       0],
[                    0,         0,         0, g_0/3 + g_1/3 - 2*g_2/3]])

In [58]:
c_mtx = sy.Matrix(params["c"])
c_mtx

Matrix([
[1,  1,  0, -1],
[1, -1, -1,  2],
[1,  0,  1, -1]])

In [59]:
se = sy.MatMul(c_mtx, bg_mtx, a_mtx, di, evaluate=True)
se

Matrix([
[d_0*g_0 + d_1*g_2 + d_2*g_1],
[d_0*g_1 + d_1*g_0 + d_2*g_2],
[d_0*g_2 + d_1*g_1 + d_2*g_0]])

In [60]:
symmetrical_cyclic_convolution(di, gi)

Matrix([
[d_0*g_0 + d_1*g_2 + d_2*g_1],
[d_0*g_1 + d_1*g_0 + d_2*g_2],
[d_0*g_2 + d_1*g_1 + d_2*g_0]])

In [61]:
bg_mtx_value = bg_mtx.subs({k[0]: v for k, v in zip(gi.tolist(), kernel[0])})
bg_mtx_value

Matrix([
[4/3, 0, 0,   0],
[  0, 0, 0,   0],
[  0, 0, 1,   0],
[  0, 0, 0, 1/3]])

In [62]:
sy.MatMul(c_mtx, bg_mtx_value, a_mtx, di)

Matrix([
[1,  1,  0, -1],
[1, -1, -1,  2],
[1,  0,  1, -1]])*Matrix([
[4/3, 0, 0,   0],
[  0, 0, 0,   0],
[  0, 0, 1,   0],
[  0, 0, 0, 1/3]])*Matrix([
[1, 1,  1],
[1, 0, -1],
[0, 1, -1],
[1, 1, -2]])*Matrix([
[d_0],
[d_1],
[d_2]])

In [63]:
d = [0, 0] + [data[0, 0]]
d

[0, 0, 0]

In [64]:
c_mtx * bg_mtx_value * a_mtx * sy.Matrix(d)

Matrix([
[0],
[0],
[0]])

In [65]:
d = [0] + data[0, 0:2].tolist()
d

[0, 0, 1]

In [66]:
c_mtx * bg_mtx_value * a_mtx * sy.Matrix(d)

Matrix([
[2],
[1],
[1]])

In [67]:
d = data[0].tolist()
d

[0, 1, 2, 3, 4]

In [68]:
d[0: 3][::-1]

[2, 1, 0]

In [69]:
d1 = d[0: 3][::-1]
d1

[2, 1, 0]

In [70]:
c_mtx * bg_mtx_value * a_mtx * sy.Matrix(d1)

Matrix([
[3],
[5],
[4]])

In [85]:
symmetrical_cyclic_convolution(kernel[0], d1[0:3])

Matrix([
[3],
[5],
[4]])

In [72]:
symmetrical_cyclic_convolution(kernel[0], d1[1:4])

Matrix([
[1],
[2],
[1]])

In [73]:
symmetrical_cyclic_convolution(kernel[0], d1[2:])

Matrix([
[0],
[0],
[0]])

In [74]:
# data[0, 0:3].flip()

In [75]:
c_mtx * bg_mtx_value * a_mtx * sy.Matrix(d[0:3])

Matrix([
[5],
[3],
[4]])

In [76]:
c_mtx * bg_mtx_value * a_mtx * sy.Matrix(d[1:4])

Matrix([
[9],
[7],
[8]])

In [77]:
c_mtx * bg_mtx_value * a_mtx * sy.Matrix(d[2:5])

Matrix([
[13],
[11],
[12]])

In [80]:
sy.Matrix(np.convolve(d, kernel[0]))

Matrix([
[ 0],
[ 1],
[ 4],
[ 8],
[12],
[11],
[ 4]])

In [83]:
symmetrical_cyclic_convolution(kernel[0], [1, 2, 3])

Matrix([
[9],
[7],
[8]])

In [99]:
x_arr = np.array(d)
size = x_arr.shape[0]
xx = np.tile(x_arr.reshape(-1), 2)
yy = np.array(kernel[0]).reshape(-1)
out = np.convolve(xx, yy)
out_clip = out[size:2 * size]
out_mtx = sy.Matrix(out_clip)

In [100]:
sy.Matrix(out_clip)

Matrix([
[11],
[ 5],
[ 4],
[ 8],
[12]])